# ForgetProof — Live Interactive Demo

This notebook shows our project live, step by step:
1. Train a model that genuinely **knows** real facts
2. **"Unlearn"** one of them (using a published method, and our own improved version)
3. **Ask it directly** — does it still know?
4. **Test whether it really forgot**, or just learned to hide it (the "relearning attack")
5. **Try it yourself** — ask any question live and compare all the models instantly

Run each cell in order (Shift+Enter). The first few cells take a few minutes on the GPU —
talk through what's happening while they run. The last cell is instant and re-runnable:
change the question and run it again as many times as you like, live.

In [ ]:
import subprocess, os, sys
subprocess.run(["pip", "install", "-q", "bitsandbytes"], check=True)

REPO_DIR = "/kaggle/working/forgetproof"
if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "https://github.com/vinay-reddaboina/forgetproof.git", REPO_DIR], check=True)
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
subprocess.run(["pip", "install", "-q", "-r", "requirements.txt"], check=True)
print("ready")

## Step 1 — Train a model that knows things

We fine-tune GPT-2 (124M parameters) on **TOFU**, a dataset of Q&A about 200 fictional
authors — so the model genuinely *knows* facts about all of them, the same way a real LLM
learns facts from its training data. This takes about 5–8 minutes on this GPU.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from src.data_utils import load_tofu, load_tofu_full
from src.finetune_target import finetune_target_model

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

MODEL_NAME = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.pad_token or tokenizer.eos_token

forget_ds, retain_ds = load_tofu("forget10")
full_ds = load_tofu_full()

target_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float16)
target_model = finetune_target_model(target_model, tokenizer, full_ds, device=device)
target_model.to("cpu")
torch.cuda.empty_cache()
print("target model trained")

## Step 2 — Ask it something real

Let's pick one real question about one author and ask the model directly.

In [ ]:
from src.eval.metrics import sequence_loss
from src.live_demo import generate_answer

DEMO_IDX = 160  # change this any time to try a different author/fact
example = forget_ds[DEMO_IDX]
question, answer = example["question"], example["answer"]
print("Question:", question)
print("True answer:", answer)

target_model.to(device)
loss = sequence_loss(target_model, tokenizer, question, answer, device)
print(f"\nconfusion score: {loss:.2f}  (lower = more confident/correct)")
print("model says:", generate_answer(target_model, tokenizer, question, device))
target_model.to("cpu")
torch.cuda.empty_cache()

## Step 3 — Make it "forget" this data (unlearning)

Now we load two **real, already-trained** unlearned versions of this exact model, from our
actual experiment:

- **npo** — a standard unlearning method, trained to stop answering forget-set questions
- **npo_mcu** — the same thing, plus **MCU**, a technique from a published paper
  (Xiao et al., arXiv:2605.11685) that's supposed to make the forgetting harder to undo

In [ ]:
import glob

ckpt_candidates = sorted(glob.glob("/kaggle/input/**/npo.pt", recursive=True))
assert ckpt_candidates, "npo.pt not found -- make sure the main kernel is attached as an input"
npo_path = ckpt_candidates[0]
npo_mcu_path = npo_path.replace("npo.pt", "npo_mcu.pt")

def load_checkpoint(path):
    m = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float16)
    m.load_state_dict(torch.load(path, map_location="cpu"))
    return m

npo_model = load_checkpoint(npo_path)
npo_mcu_model = load_checkpoint(npo_mcu_path)
print("loaded the real unlearned checkpoints")

## Step 4 — Ask the "forgetful" models the same question

Watch the confidence score — does it still know the answer, or has it really lost it?

In [ ]:
for name, model in [("npo", npo_model), ("npo_mcu", npo_mcu_model)]:
    model.to(device)
    loss = sequence_loss(model, tokenizer, question, answer, device)
    gen = generate_answer(model, tokenizer, question, device)
    print(f"[{name}]  confusion score: {loss:.2f}")
    print("  says:", gen)
    model.to("cpu")
    torch.cuda.empty_cache()

## Step 5 — Did it really forget, or just hide it?

This is the real test: give the "forgetful" model a *tiny* bit of extra training on
**320 other questions** (never this one), then ask again. If the confidence comes back,
the fact wasn't erased — it was just suppressed.

In [ ]:
from src.live_demo import relearning_attack_live
from src.data_utils import held_out_split

relearn_ds, holdout_ds = held_out_split(forget_ds, holdout_frac=0.2, seed=42)
print(f"retraining on {len(relearn_ds)} OTHER questions -- never the one we're testing")

attack_results = {}
for name, model in [("npo", npo_model), ("npo_mcu", npo_mcu_model)]:
    pre_loss = sequence_loss(model.to(device), tokenizer, question, answer, device)
    attacked = relearning_attack_live(model, tokenizer, relearn_ds, device)
    post_loss = sequence_loss(attacked, tokenizer, question, answer, device)
    recovery = max(0.0, min(1.0, (pre_loss - post_loss) / max(pre_loss, 1e-6)))
    print(f"[{name}] confusion {pre_loss:.2f} -> {post_loss:.2f}  ({recovery*100:.0f}% recovered)")
    print("  now says:", generate_answer(attacked, tokenizer, question, device))
    attack_results[name] = recovery
    model.to("cpu"); del attacked; torch.cuda.empty_cache()

## Step 6 — Our improvement: does tuning MCU actually help?

The paper's own default settings for MCU are one fixed choice. We ran a 60-setting search
and found a better one. Let's train that version **live, right now**, and run the same
test on it.

In [ ]:
import copy
from src.unlearn.npo_mcu import unlearn_npo_mcu

print("training npo_mcu with our tuned settings (k=32, layer=2)...")
npo_mcu_tuned = unlearn_npo_mcu(
    copy.deepcopy(target_model.to(device)), tokenizer, forget_ds, retain_ds,
    layer_idx=2, mcu_k=32, device=device,
)
target_model.to("cpu")

pre_loss = sequence_loss(npo_mcu_tuned, tokenizer, question, answer, device)
print(f"\n[npo_mcu tuned] confusion before attack: {pre_loss:.2f}")
attacked = relearning_attack_live(npo_mcu_tuned, tokenizer, relearn_ds, device)
post_loss = sequence_loss(attacked, tokenizer, question, answer, device)
recovery = max(0.0, min(1.0, (pre_loss - post_loss) / max(pre_loss, 1e-6)))
print(f"[npo_mcu tuned] confusion after attack:  {post_loss:.2f}  ({recovery*100:.0f}% recovered)")
attack_results["npo_mcu_tuned"] = recovery
npo_mcu_tuned.to("cpu"); del attacked; torch.cuda.empty_cache()

print("\n" + "=" * 60)
print("SUMMARY -- lower % recovered = more robust = better")
print("=" * 60)
for name, rec in attack_results.items():
    print(f"  {name:16s} {rec*100:5.0f}% recovered")

## Step 7 — Try it yourself, live

Pick any question below (change `MY_QUESTION_IDX`, any number from 0–399) and run this
cell — it re-uses the models already loaded above, so it's instant. Ask the teacher to
pick a number.

In [ ]:
MY_QUESTION_IDX = 25   # <-- change this number and re-run the cell, as many times as you like

ex = forget_ds[MY_QUESTION_IDX]
print("Question:", ex["question"])
print("True answer:", ex["answer"])
print()

models_to_try = {
    "target (knows everything)": target_model,
    "npo (unlearned)": npo_model,
    "npo_mcu (paper's settings)": npo_mcu_model,
    "npo_mcu (our tuned settings)": npo_mcu_tuned,
}
for name, model in models_to_try.items():
    model.to(device)
    print(f"[{name}]")
    print("  says:", generate_answer(model, tokenizer, ex["question"], device))
    model.to("cpu")
torch.cuda.empty_cache()